In [ ]:
# -*- coding: utf-8 -*-

import os
import re
import html
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw

# =========================================================
# 0. Paths
# =========================================================
OLD_PROTO_FREQ_CSV = "./figs_data/fig3_e.csv"

OUT_DIR = "./SI_figs"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_SVG = os.path.join(OUT_DIR, "SI_old_prototype_shared_motifs_matrix.svg")
OUT_CSV = os.path.join(OUT_DIR, "SI_old_prototype_shared_motifs_matrix.csv")


# =========================================================
# 1. SMARTS patterns
# =========================================================
SMARTS_PATTERNS = {
    "tetra-alkoxy borate": "[#5](-[#8])(-[#8])(-[#8])-[#8]",
    "tri-alkoxy borate": "[#5;D3;+0](-[#8])(-[#8])-[#8]",
    "boron_oxalate": "[#5]1[#8][#6](=[#8])[#6](=[#8])[#8]1",
    "BF3-like motif": "[#5](-[#9])(-[#9])-[#9]",
    "fluorinated alkoxy": "[#8]-[#6]-[#6](-[#9])(-[#9])-[#9]",
    "C-B(OR)2 motif": "[#6]-[#5;D3;+0](-[#8]-[#6])-[#8]-[#6]",
    "pentafluorophenyl group": "[c;D3]1[c]([#9])[c]([#9])[c]([#9])[c]([#9])[c]1[#9]",
    "catechol boronate": "[#5]1-[#8]-c2ccccc2-[#8]-1",
}

MOTIF_COLORS = {
    "tetra-alkoxy borate": "#B2182B",
    "tri-alkoxy borate": "#D06A5C",
    "boron_oxalate": "#E6A68B",
    "BF3-like motif": "#E9CABD",
    "fluorinated alkoxy": "#A8C4D4",
    "C-B(OR)2 motif": "#5F8FB8",
    "pentafluorophenyl group": "#3B6EA8",
    "catechol boronate": "#11519B",
}

motif_order = list(SMARTS_PATTERNS.keys())


# =========================================================
# 2. Representative SMILES for clean motif drawings
# =========================================================
REPRESENTATIVE_SMILES = {
    "tetra-alkoxy borate": "B(OC)(OC)(OC)OC",
    "tri-alkoxy borate": "B(OC)(OC)OC",
    "boron_oxalate": "O=C1OBOC(=O)1",
    "BF3-like motif": "B(F)(F)F",
    "fluorinated alkoxy": "COCC(F)(F)F",
    "C-B(OR)2 motif": "CB(OC)OC",
    "pentafluorophenyl group": "Fc1c(F)c(F)c(F)c(F)c1",
    "catechol boronate": "B1Oc2ccccc2O1",
}


# =========================================================
# 3. Helper functions
# =========================================================
def sort_proto_labels(labels):
    return sorted(
        labels,
        key=lambda x: int("".join([c for c in str(x) if c.isdigit()]) or 999)
    )


def wrap_text(text, max_chars=17):
    words = text.split()
    lines = []
    current = ""

    for w in words:
        if len(current) + len(w) + 1 <= max_chars:
            current = (current + " " + w).strip()
        else:
            if current:
                lines.append(current)
            current = w

    if current:
        lines.append(current)

    return lines


def mol_to_inner_svg(mol, width=210, height=120):
    drawer = Draw.MolDraw2DSVG(width, height)
    opts = drawer.drawOptions()
    opts.clearBackground = False
    opts.bondLineWidth = 2.0
    opts.minFontSize = 12
    opts.maxFontSize = 18

    Draw.rdMolDraw2D.PrepareAndDrawMolecule(drawer, mol)
    drawer.FinishDrawing()

    svg = drawer.GetDrawingText()
    svg = re.sub(r"<\?xml.*?\?>", "", svg, flags=re.S)
    svg = re.sub(r"<!DOCTYPE.*?>", "", svg, flags=re.S)
    svg = re.sub(r"<svg[^>]*>", "", svg, flags=re.S)
    svg = svg.replace("</svg>", "")

    return svg.strip()


def get_motif_mol(motif):
    smi = REPRESENTATIVE_SMILES.get(motif, None)

    mol = None
    if smi is not None:
        mol = Chem.MolFromSmiles(smi)

    if mol is None:
        mol = Chem.MolFromSmarts(SMARTS_PATTERNS[motif])

    if mol is None:
        raise ValueError(f"Cannot generate molecule for motif: {motif}")

    return mol


# =========================================================
# 4. Load motif frequency data
# =========================================================
freq_df = pd.read_csv(OLD_PROTO_FREQ_CSV)

required_cols = [
    "prototype",
    "motif",
    "smarts",
    "n_molecules_in_prototype",
    "motif_count",
    "motif_frequency",
]

for col in required_cols:
    if col not in freq_df.columns:
        raise ValueError(f"fig3_e.csv 缺少必要列: {col}")

freq_df["prototype"] = freq_df["prototype"].astype(str)
freq_df["motif"] = freq_df["motif"].astype(str)
freq_df["motif_frequency"] = freq_df["motif_frequency"].astype(float)

proto_order = sort_proto_labels(freq_df["prototype"].unique())

matrix_df = (
    freq_df
    .pivot_table(
        index="prototype",
        columns="motif",
        values="motif_frequency",
        aggfunc="mean",
        fill_value=0.0
    )
    .reindex(index=proto_order, columns=motif_order)
)

matrix_df.to_csv(OUT_CSV)

print("\nMotif frequency matrix:")
print(matrix_df.round(3))


# =========================================================
# 5. Pre-generate motif SVG drawings
# =========================================================
motif_svg_dict = {}

for motif in motif_order:
    mol = get_motif_mol(motif)
    motif_svg_dict[motif] = mol_to_inner_svg(
        mol,
        width=210,
        height=120
    )


# =========================================================
# 6. SVG layout settings
# =========================================================
cell_w = 260
cell_h = 215

left_margin = 170
top_margin = 250
right_margin = 40
bottom_margin = 90

n_rows = len(proto_order)
n_cols = len(motif_order)

svg_w = left_margin + n_cols * cell_w + right_margin
svg_h = top_margin + n_rows * cell_h + bottom_margin

font_family = "Arial"


# =========================================================
# 7. Compose SVG
# =========================================================
svg_parts = []

svg_parts.append(
    f'''<svg xmlns="http://www.w3.org/2000/svg" width="{svg_w}" height="{svg_h}" viewBox="0 0 {svg_w} {svg_h}">'''
)

svg_parts.append(
    f'''<rect x="0" y="0" width="{svg_w}" height="{svg_h}" fill="white"/>'''
)

# Title
svg_parts.append(
    f'''
    <text x="{svg_w / 2}" y="55"
          text-anchor="middle"
          font-family="{font_family}"
          font-size="34"
          font-weight="700"
          fill="#222222">
        Shared molecular motifs across old prototypes
    </text>
    '''
)

svg_parts.append(
    f'''
    <text x="{svg_w / 2}" y="95"
          text-anchor="middle"
          font-family="{font_family}"
          font-size="20"
          fill="#555555">
        Values indicate motif frequency within each prototype
    </text>
    '''
)

# Column headers
for j, motif in enumerate(motif_order):
    x = left_margin + j * cell_w
    cx = x + cell_w / 2
    color = MOTIF_COLORS[motif]

    svg_parts.append(
        f'''
        <rect x="{x + 12}" y="125" width="{cell_w - 24}" height="8"
              fill="{color}" rx="4" ry="4"/>
        '''
    )

    lines = wrap_text(motif, max_chars=18)

    for k, line in enumerate(lines):
        svg_parts.append(
            f'''
            <text x="{cx}" y="{158 + k * 22}"
                  text-anchor="middle"
                  font-family="{font_family}"
                  font-size="17"
                  font-weight="600"
                  fill="#222222">
                {html.escape(line)}
            </text>
            '''
        )


# Row labels
for i, proto in enumerate(proto_order):
    y = top_margin + i * cell_h
    cy = y + cell_h / 2

    svg_parts.append(
        f'''
        <text x="{left_margin - 42}" y="{cy + 8}"
              text-anchor="middle"
              font-family="{font_family}"
              font-size="28"
              font-weight="700"
              fill="#222222">
            {html.escape(proto)}
        </text>
        '''
    )


# Cells
for i, proto in enumerate(proto_order):
    for j, motif in enumerate(motif_order):
        freq = float(matrix_df.loc[proto, motif])
        percent = freq * 100

        x = left_margin + j * cell_w
        y = top_margin + i * cell_h

        color = MOTIF_COLORS[motif]

        if freq > 0:
            opacity = 0.12 + 0.35 * min(freq, 1.0)
            border_color = color
            border_opacity = 0.65
        else:
            opacity = 0.035
            border_color = "#D9D9D9"
            border_opacity = 0.65

        svg_parts.append(
            f'''
            <rect x="{x + 8}" y="{y + 8}" width="{cell_w - 16}" height="{cell_h - 16}"
                  fill="{color if freq > 0 else '#F2F2F2'}"
                  fill-opacity="{opacity}"
                  stroke="{border_color}"
                  stroke-opacity="{border_opacity}"
                  stroke-width="1.2"
                  rx="14" ry="14"/>
            '''
        )

        if freq > 0:
            # molecule structure: 缩小并上移
            mol_x = x + 25
            mol_y = y + 22

            svg_parts.append(
                f'''
                <g transform="translate({mol_x},{mol_y})">
                    {motif_svg_dict[motif]}
                </g>
                '''
            )

            # percentage background
            svg_parts.append(
                f'''
                <rect x="{x + 55}" y="{y + cell_h - 52}"
                      width="{cell_w - 110}" height="34"
                      fill="white"
                      fill-opacity="0.78"
                      rx="8" ry="8"/>
                '''
            )

            # percentage text
            svg_parts.append(
                f'''
                <text x="{x + cell_w / 2}" y="{y + cell_h - 28}"
                      text-anchor="middle"
                      font-family="{font_family}"
                      font-size="23"
                      font-weight="700"
                      fill="{color}">
                    {percent:.1f}%
                </text>
                '''
            )

        else:
            svg_parts.append(
                f'''
                <text x="{x + cell_w / 2}" y="{y + cell_h / 2 + 8}"
                      text-anchor="middle"
                      font-family="{font_family}"
                      font-size="24"
                      font-weight="500"
                      fill="#BBBBBB">
                    —
                </text>
                '''
            )


# Footer note
svg_parts.append(
    f'''
    <text x="{left_margin}" y="{svg_h - 35}"
          text-anchor="start"
          font-family="{font_family}"
          font-size="16"
          fill="#666666">
        Motif frequencies were calculated as motif_count / n_molecules_in_prototype.
    </text>
    '''
)

svg_parts.append("</svg>")

with open(OUT_SVG, "w", encoding="utf-8") as f:
    f.write("\n".join(svg_parts))

print("\nSaved SVG:", OUT_SVG)
print("Saved CSV:", OUT_CSV)


Motif frequency matrix:
motif      tetra-alkoxy borate  tri-alkoxy borate  boron_oxalate  \
prototype                                                          
P1                       0.000              0.125          0.000   
P2                       0.250              0.083          0.167   
P3                       0.571              0.071          0.643   
P4                       0.421              0.053          0.000   
P5                       0.000              0.318          0.000   
P6                       0.071              0.643          0.000   
P7                       0.042              0.708          0.000   

motif      BF3-like motif  fluorinated alkoxy  C-B(OR)2 motif  \
prototype                                                       
P1                  0.188               0.000           0.562   
P2                  0.000               0.000           0.167   
P3                  0.000               0.000           0.071   
P4                  0.000            

[17:31:23] Explicit valence for atom # 0 B, 4, is greater than permitted


In [2]:
# -*- coding: utf-8 -*-

import os
import re
import html
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw


# =========================================================
# 0. Paths
# =========================================================
OLD_PROTO_FREQ_CSV = "./figs_data/fig3_e.csv"

OUT_DIR = "./SI_figs"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_SVG = os.path.join(OUT_DIR, "SI_old_prototype_shared_motifs_matrix.svg")
OUT_CSV = os.path.join(OUT_DIR, "SI_old_prototype_shared_motifs_matrix.csv")

# 单独导出 8 个 shared motif SVG 的文件夹
MOTIF_SVG_DIR = os.path.join(OUT_DIR, "shared_motif_individual_svgs")
os.makedirs(MOTIF_SVG_DIR, exist_ok=True)


# =========================================================
# 1. SMARTS patterns
# =========================================================
SMARTS_PATTERNS = {
    "tetra-alkoxy borate": "[#5](-[#8])(-[#8])(-[#8])-[#8]",
    "tri-alkoxy borate": "[#5;D3;+0](-[#8])(-[#8])-[#8]",
    "boron_oxalate": "[#5]1[#8][#6](=[#8])[#6](=[#8])[#8]1",
    "BF3-like motif": "[#5](-[#9])(-[#9])-[#9]",
    "fluorinated alkoxy": "[#8]-[#6]-[#6](-[#9])(-[#9])-[#9]",
    "C-B(OR)2 motif": "[#6]-[#5;D3;+0](-[#8]-[#6])-[#8]-[#6]",
    "pentafluorophenyl group": "[c;D3]1[c]([#9])[c]([#9])[c]([#9])[c]([#9])[c]1[#9]",
    "catechol boronate": "[#5]1-[#8]-c2ccccc2-[#8]-1",
}

MOTIF_COLORS = {
    "tetra-alkoxy borate": "#B2182B",
    "tri-alkoxy borate": "#D06A5C",
    "boron_oxalate": "#E6A68B",
    "BF3-like motif": "#E9CABD",
    "fluorinated alkoxy": "#A8C4D4",
    "C-B(OR)2 motif": "#5F8FB8",
    "pentafluorophenyl group": "#3B6EA8",
    "catechol boronate": "#11519B",
}

motif_order = list(SMARTS_PATTERNS.keys())


# =========================================================
# 2. Representative SMILES for clean motif drawings
# =========================================================
REPRESENTATIVE_SMILES = {
    "tetra-alkoxy borate": "B(OC)(OC)(OC)OC",
    "tri-alkoxy borate": "B(OC)(OC)OC",
    "boron_oxalate": "O=C1OBOC(=O)1",
    "BF3-like motif": "B(F)(F)F",
    "fluorinated alkoxy": "COCC(F)(F)F",
    "C-B(OR)2 motif": "CB(OC)OC",
    "pentafluorophenyl group": "Fc1c(F)c(F)c(F)c(F)c1",
    "catechol boronate": "B1Oc2ccccc2O1",
}


# =========================================================
# 3. Helper functions
# =========================================================
def sort_proto_labels(labels):
    return sorted(
        labels,
        key=lambda x: int("".join([c for c in str(x) if c.isdigit()]) or 999)
    )


def wrap_text(text, max_chars=17):
    words = text.split()
    lines = []
    current = ""

    for w in words:
        if len(current) + len(w) + 1 <= max_chars:
            current = (current + " " + w).strip()
        else:
            if current:
                lines.append(current)
            current = w

    if current:
        lines.append(current)

    return lines


def safe_filename(text):
    """
    Convert motif name to a safe SVG filename.
    """
    text = text.replace(" ", "_")
    text = text.replace("/", "_")
    text = text.replace("\\", "_")
    text = text.replace("(", "")
    text = text.replace(")", "")
    text = text.replace("+", "plus")
    text = text.replace("-", "_")
    text = re.sub(r"[^A-Za-z0-9_]+", "_", text)
    text = re.sub(r"_+", "_", text)
    text = text.strip("_")
    return text



def remove_svg_background(svg):
    """
    Remove white/opaque background rectangles from RDKit SVG output.
    This makes both embedded and standalone motif SVGs transparent.
    """
    # Remove simple white background rectangles, e.g. fill="white" or fill="#FFFFFF"
    svg = re.sub(
        r"<rect\b[^>]*(?:fill\s*=\s*[\"'](?:white|#fff|#ffffff|rgb\(255,\s*255,\s*255\))[\"'])[^>]*/?>",
        "",
        svg,
        flags=re.I | re.S,
    )

    # Remove RDKit-style white background rectangles defined in style attribute
    svg = re.sub(
        r"<rect\b[^>]*style\s*=\s*[\"'][^\"']*fill\s*:\s*(?:white|#fff|#ffffff|rgb\(255,\s*255,\s*255\))[^\"']*[\"'][^>]*/?>",
        "",
        svg,
        flags=re.I | re.S,
    )

    # In case an SVG renderer uses CSS background declarations
    svg = re.sub(
        r"background(?:-color)?\s*:\s*(?:white|#fff|#ffffff)\s*;?",
        "",
        svg,
        flags=re.I,
    )

    return svg


def get_motif_mol(motif):
    smi = REPRESENTATIVE_SMILES.get(motif, None)

    mol = None
    if smi is not None:
        mol = Chem.MolFromSmiles(smi)

    if mol is None:
        mol = Chem.MolFromSmarts(SMARTS_PATTERNS[motif])

    if mol is None:
        raise ValueError(f"Cannot generate molecule for motif: {motif}")

    Chem.rdDepictor.Compute2DCoords(mol)

    return mol


def mol_to_inner_svg(mol, width=210, height=120):
    """
    Generate inner SVG content without outer <svg> tag.
    Used for embedding molecule drawings into the large motif matrix SVG.
    """
    drawer = Draw.MolDraw2DSVG(width, height)
    opts = drawer.drawOptions()
    opts.clearBackground = False
    opts.bondLineWidth = 2.0
    opts.minFontSize = 12
    opts.maxFontSize = 18

    Draw.rdMolDraw2D.PrepareAndDrawMolecule(drawer, mol)
    drawer.FinishDrawing()

    svg = drawer.GetDrawingText()
    svg = remove_svg_background(svg)
    svg = re.sub(r"<\?xml.*?\?>", "", svg, flags=re.S)
    svg = re.sub(r"<!DOCTYPE.*?>", "", svg, flags=re.S)
    svg = re.sub(r"<svg[^>]*>", "", svg, flags=re.S)
    svg = svg.replace("</svg>", "")

    return svg.strip()


def mol_to_full_svg(mol, width=360, height=260):
    """
    Generate a standalone SVG file for a motif molecule.
    This version uses a transparent background and a clean viewBox,
    suitable for tables, AI, PowerPoint, or SI figure assembly.
    """
    drawer = Draw.MolDraw2DSVG(width, height)
    opts = drawer.drawOptions()

    opts.clearBackground = False
    opts.bondLineWidth = 2.2
    opts.minFontSize = 14
    opts.maxFontSize = 24

    Draw.rdMolDraw2D.PrepareAndDrawMolecule(drawer, mol)
    drawer.FinishDrawing()

    svg = drawer.GetDrawingText()
    svg = remove_svg_background(svg)

    # Make the standalone SVG explicitly transparent for common SVG renderers.
    svg = re.sub(
        r"<svg\b",
        '<svg style="background: transparent;"',
        svg,
        count=1,
        flags=re.I,
    )

    return svg


def export_single_motif_svg(motif, mol, out_dir, width=360, height=260):
    """
    Export one motif as a standalone SVG.
    """
    filename = safe_filename(motif) + ".svg"
    out_path = os.path.join(out_dir, filename)

    svg = mol_to_full_svg(
        mol,
        width=width,
        height=height
    )

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(svg)

    return out_path


# =========================================================
# 4. Load motif frequency data
# =========================================================
freq_df = pd.read_csv(OLD_PROTO_FREQ_CSV)

required_cols = [
    "prototype",
    "motif",
    "smarts",
    "n_molecules_in_prototype",
    "motif_count",
    "motif_frequency",
]

for col in required_cols:
    if col not in freq_df.columns:
        raise ValueError(f"fig3_e.csv 缺少必要列: {col}")

freq_df["prototype"] = freq_df["prototype"].astype(str)
freq_df["motif"] = freq_df["motif"].astype(str)
freq_df["motif_frequency"] = freq_df["motif_frequency"].astype(float)

proto_order = sort_proto_labels(freq_df["prototype"].unique())

matrix_df = (
    freq_df
    .pivot_table(
        index="prototype",
        columns="motif",
        values="motif_frequency",
        aggfunc="mean",
        fill_value=0.0
    )
    .reindex(index=proto_order, columns=motif_order)
)

matrix_df.to_csv(OUT_CSV)

print("\nMotif frequency matrix:")
print(matrix_df.round(3))


# =========================================================
# 5. Pre-generate motif SVG drawings
#    同时单独导出 8 个 shared motif SVG
# =========================================================
motif_svg_dict = {}
single_svg_paths = []

print("\nGenerating motif SVG drawings...")

for motif in motif_order:
    mol = get_motif_mol(motif)

    # 用于大矩阵图内部嵌入
    motif_svg_dict[motif] = mol_to_inner_svg(
        mol,
        width=210,
        height=120
    )

    # 单独导出为 SVG 图片
    single_svg_path = export_single_motif_svg(
        motif=motif,
        mol=mol,
        out_dir=MOTIF_SVG_DIR,
        width=360,
        height=260
    )

    single_svg_paths.append(single_svg_path)
    print(f"Saved individual motif SVG: {single_svg_path}")


# =========================================================
# 6. SVG layout settings
# =========================================================
cell_w = 260
cell_h = 215

left_margin = 170
top_margin = 250
right_margin = 90
bottom_margin = 90

n_rows = len(proto_order)
n_cols = len(motif_order)

svg_w = left_margin + n_cols * cell_w + right_margin
svg_h = top_margin + n_rows * cell_h + bottom_margin

font_family = "Arial"


# =========================================================
# 7. Compose SVG matrix figure
# =========================================================
svg_parts = []

svg_parts.append(
    f'''<svg xmlns="http://www.w3.org/2000/svg" width="{svg_w}" height="{svg_h}" viewBox="0 0 {svg_w} {svg_h}" style="background: transparent;">'''
)

# Do not add an outer background rectangle.
# Keeping this absent makes the final matrix SVG background transparent.

# Title
svg_parts.append(
    f'''
    <text x="{svg_w / 2}" y="55"
          text-anchor="middle"
          font-family="{font_family}"
          font-size="34"
          font-weight="700"
          fill="#222222">
        Shared molecular motifs across old prototypes
    </text>
    '''
)

svg_parts.append(
    f'''
    <text x="{svg_w / 2}" y="95"
          text-anchor="middle"
          font-family="{font_family}"
          font-size="20"
          fill="#555555">
        Values indicate motif frequency within each prototype
    </text>
    '''
)

# Column headers
for j, motif in enumerate(motif_order):
    x = left_margin + j * cell_w
    cx = x + cell_w / 2
    color = MOTIF_COLORS[motif]

    svg_parts.append(
        f'''
        <rect x="{x + 12}" y="125" width="{cell_w - 24}" height="8"
              fill="{color}" rx="4" ry="4"/>
        '''
    )

    lines = wrap_text(motif, max_chars=18)

    for k, line in enumerate(lines):
        svg_parts.append(
            f'''
            <text x="{cx}" y="{158 + k * 22}"
                  text-anchor="middle"
                  font-family="{font_family}"
                  font-size="17"
                  font-weight="600"
                  fill="#222222">
                {html.escape(line)}
            </text>
            '''
        )


# Row labels
for i, proto in enumerate(proto_order):
    y = top_margin + i * cell_h
    cy = y + cell_h / 2

    svg_parts.append(
        f'''
        <text x="{left_margin - 42}" y="{cy + 8}"
              text-anchor="middle"
              font-family="{font_family}"
              font-size="28"
              font-weight="700"
              fill="#222222">
            {html.escape(proto)}
        </text>
        '''
    )


# Cells
for i, proto in enumerate(proto_order):
    for j, motif in enumerate(motif_order):
        freq = float(matrix_df.loc[proto, motif])
        percent = freq * 100

        x = left_margin + j * cell_w
        y = top_margin + i * cell_h

        color = MOTIF_COLORS[motif]

        if freq > 0:
            opacity = 0.12 + 0.35 * min(freq, 1.0)
            border_color = color
            border_opacity = 0.65
        else:
            opacity = 0.035
            border_color = "#D9D9D9"
            border_opacity = 0.65

        svg_parts.append(
            f'''
            <rect x="{x + 8}" y="{y + 8}" width="{cell_w - 16}" height="{cell_h - 16}"
                  fill="{color if freq > 0 else '#F2F2F2'}"
                  fill-opacity="{opacity}"
                  stroke="{border_color}"
                  stroke-opacity="{border_opacity}"
                  stroke-width="1.2"
                  rx="14" ry="14"/>
            '''
        )

        if freq > 0:
            # Molecule structure
            mol_x = x + 25
            mol_y = y + 22

            svg_parts.append(
                f'''
                <g transform="translate({mol_x},{mol_y})">
                    {motif_svg_dict[motif]}
                </g>
                '''
            )

            # Percentage background
            # Do not draw a white label box, so the exported SVG remains transparent.
            # If you later want a label box, use fill="none" instead of fill="white".
            # svg_parts.append(
            #     f'''
            #     <rect x="{x + 55}" y="{y + cell_h - 52}"
            #           width="{cell_w - 110}" height="34"
            #           fill="none"
            #           fill-opacity="0"
            #           rx="8" ry="8"/>
            #     '''
            # )

            # Percentage text
            svg_parts.append(
                f'''
                <text x="{x + cell_w / 2}" y="{y + cell_h - 28}"
                      text-anchor="middle"
                      font-family="{font_family}"
                      font-size="23"
                      font-weight="700"
                      fill="{color}">
                    {percent:.1f}%
                </text>
                '''
            )

        else:
            svg_parts.append(
                f'''
                <text x="{x + cell_w / 2}" y="{y + cell_h / 2 + 8}"
                      text-anchor="middle"
                      font-family="{font_family}"
                      font-size="24"
                      font-weight="500"
                      fill="#BBBBBB">
                    —
                </text>
                '''
            )


# Footer note
svg_parts.append(
    f'''
    <text x="{left_margin}" y="{svg_h - 35}"
          text-anchor="start"
          font-family="{font_family}"
          font-size="16"
          fill="#666666">
        Motif frequencies were calculated as motif_count / n_molecules_in_prototype.
    </text>
    '''
)

svg_parts.append("</svg>")

with open(OUT_SVG, "w", encoding="utf-8") as f:
    f.write("\n".join(svg_parts))


# =========================================================
# 8. Final print
# =========================================================
print("\nSaved matrix SVG:", OUT_SVG)
print("Saved matrix CSV:", OUT_CSV)

print("\nSaved individual motif SVGs:")
for p in single_svg_paths:
    print(p)


Motif frequency matrix:
motif      tetra-alkoxy borate  tri-alkoxy borate  boron_oxalate  \
prototype                                                          
P1                       0.000              0.125          0.000   
P2                       0.250              0.083          0.167   
P3                       0.571              0.071          0.643   
P4                       0.421              0.053          0.000   
P5                       0.000              0.318          0.000   
P6                       0.071              0.643          0.000   
P7                       0.042              0.708          0.000   

motif      BF3-like motif  fluorinated alkoxy  C-B(OR)2 motif  \
prototype                                                       
P1                  0.188               0.000           0.562   
P2                  0.000               0.000           0.167   
P3                  0.000               0.000           0.071   
P4                  0.000            

[17:35:50] Explicit valence for atom # 0 B, 4, is greater than permitted
